In [1]:
from src.OrganoidMesh import OrganoidMesh
from src.mesh_analysis import *
from src.nonlocal_correlations import *
from src.cell_graph_functions import *
from src.utils import *
from src.crypt_extraction import extract_all_crypts, compute_curvature_proxy
from src.organoid_plotting import *

In [2]:
def load_cell_graph_from_npz(data: np.lib.npyio.NpzFile) -> nx.Graph:
    """
    Reconstruct a NetworkX graph from edges and node count stored in an NPZ file.
    This matches the new preprocessing format where `edges` and `n_nodes`
    are explicitly saved.
    """
    if "graph_edges" not in data.files or "graph_n_nodes" not in data.files:
        raise KeyError("NPZ file must contain 'edges' and 'n_nodes' to load the cell graph.")

    edges = data["graph_edges"]
    n_nodes = int(data["graph_n_nodes"])

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))

    if edges.size > 0:
        edges = edges.reshape(-1, 2)
        G.add_edges_from(edges.tolist())

    return G


In [3]:
data_dir = '../NicoleData/20250929/fractal_output'

timepoint = "day4p5"
zarr_name = "r0.zarr"
well = "A06"
round_name = "0_fused_zillum_registered"
organoid_id = 31 # 11 #20 # 31 #19 # example

path = f"{data_dir}/{timepoint}/{zarr_name}/{well[0]}/{well[1:]}/{round_name}/meshes/nnorg_linked_multi_annotated_class/{organoid_id}.vtp"


mesh = OrganoidMesh()
mesh.load_mesh_from_file(path)
mesh.align_with_pca()
_ = mesh.compute_spectral_coefficients(lmax=15)    # must populate m.eigvals, m.eigvecs

print(mesh.v.shape)

cell_graph = build_cell_graph(mesh)

[Info] Eigen-decomposition not found. Computing now...
(18001, 3)


In [4]:
HKS_times = [1.0, 2.0, 4.0, 8, 12, 16, 20, 25.0]
hks, hks_coeffs = compute_hks(mesh, t=HKS_times,) # HKS at time-scale comparable to cell size


In [5]:
cell_center_vertices = mesh.get_centroid_vertices() # get indices of patch centers
centroids, cell_areas, cell_weighted_hks, _ = mesh.compute_cell_statistics(hks) # coarse-grain

In [10]:
curvature_proxy = compute_curvature_proxy(cell_weighted_hks, HKS_times)


crypts, necks = extract_all_crypts(
    G=cell_graph,
    HKS=curvature_proxy[:,:3],
    crypt_thresh=0.6, 
    neck_thresh=-1.00,  
    growth_thresh=-0.1,
    min_region_size=8,
    verbose=False
)

print(len(crypts))
print(len(necks))


fig = plot_organoid_graph(centroids, cell_graph, curvature_proxy[:,0], node_size=2)
add_region_overlays(fig, centroids, crypts, colorscale='YlOrRd')
add_region_overlays(fig, centroids, necks, colorscale='PuBuGn')
fig.show()

5
2


In [7]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def make_crypt_subplots_3d(coords, G, crypts, z=None, cols=3, node_size=4.5, title="Crypts"):
    """
    Create a Plotly figure with a 3D subplot for each crypt.
    
    Parameters
    ----------
    coords : (N,3) array
        Node coordinates.
    G : networkx.Graph
        Full graph with nodes labeled 0..N-1.
    crypts : list of sets
        Each set contains node indices belonging to a crypt.
    z : (N,) array or None
        Optional per-node values for coloring (e.g., HKS[:,0]). If None, a flat color is used.
    cols : int
        Number of subplot columns (rows computed automatically).
    node_size : float
        Marker size for nodes.
    title : str
        Figure title.
    
    Returns
    -------
    fig : plotly.graph_objects.Figure
    """
    if len(crypts) == 0:
        return go.Figure().update_layout(title="No crypts found")

    K = len(crypts)
    rows = int(np.ceil(K / cols))
    fig = make_subplots(
        rows=rows, cols=cols,
        specs=[[{"type": "scene"} for _ in range(cols)] for __ in range(rows)],
        subplot_titles=[f"crypt {i+1} (n={len(c)})" for i, c in enumerate(crypts)]
    )

    # helper to index subplot
    def rc(i):
        r = i // cols + 1
        c = i % cols + 1
        return r, c

    XYZ = coords[:, :3]
    for i, crypt in enumerate(crypts):
        idx = np.fromiter(crypt, dtype=int)
        if idx.size == 0:
            continue
        r, c = rc(i)

        # edges within this crypt
        ex, ey, ez = [], [], []
        crypt_set = set(crypt)
        for u, v in G.edges():
            if (u in crypt_set) and (v in crypt_set):
                ex += [XYZ[u,0], XYZ[v,0], None]
                ey += [XYZ[u,1], XYZ[v,1], None]
                ez += [XYZ[u,2], XYZ[v,2], None]

        # wireframe
        if ex:
            fig.add_trace(
                go.Scatter3d(
                    x=ex, y=ey, z=ez, mode="lines",
                    line=dict(width=1.0, color="rgba(120,120,120,0.65)"),
                    opacity=1.0, hoverinfo="skip", showlegend=False
                ),
                row=r, col=c
            )

        # nodes
        if z is None:
            node_kwargs = dict(color="rgba(31,120,180,0.85)")
            showscale = False
        else:
            zvals = z[idx]
            finite = np.isfinite(zvals)
            vmax = float(np.nanmax(np.abs(zvals[finite]))) if np.any(finite) else 1.0
            if vmax == 0.0: vmax = 1.0
            node_kwargs = dict(
                color=zvals,
                colorscale="RdBu_r",
                cmin=-vmax, cmax=vmax, cmid=0.0
            )
            showscale = (i == 0)  # show one colorbar

        fig.add_trace(
            go.Scatter3d(
                x=XYZ[idx,0], y=XYZ[idx,1], z=XYZ[idx,2],
                mode="markers",
                marker=dict(size=node_size, opacity=0.95, showscale=showscale, **node_kwargs),
                hoverinfo="skip", showlegend=False
            ),
            row=r, col=c
        )

        # tidy scene (per subplot autoscale)
        mins = XYZ[idx].min(0); maxs = XYZ[idx].max(0)
        span = float(np.max(maxs - mins)); 
        if span == 0: span = 1.0
        ctr = (mins + maxs) / 2
        fig.update_scenes(
            dict(
                xaxis=dict(range=[ctr[0]-span/2, ctr[0]+span/2], showgrid=False, showticklabels=False, zeroline=False),
                yaxis=dict(range=[ctr[1]-span/2, ctr[1]+span/2], showgrid=False, showticklabels=False, zeroline=False),
                zaxis=dict(range=[ctr[2]-span/2, ctr[2]+span/2], showgrid=False, showticklabels=False, zeroline=False),
                aspectmode="cube",
            ),
            row=r, col=c
        )

    fig.update_layout(title=title, margin=dict(l=0, r=0, t=40, b=0))
    return fig


# crypts is a list[set[int]] from your pipeline; color by lowest-time HKS:
fig = make_crypt_subplots_3d(centroids, cell_graph, crypts, z=curvature_proxy[:,0], cols=3, title="All crypts")
fig.show()